In [ ]:

%pip install -q -U "crewai[google-genai]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:

import os
from google.colab import userdata

from crewai import LLM
from crewai.flow.flow import Flow, start, listen, router
from pydantic import BaseModel, Field
from crewai.flow.flow import or_


GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")


Gemini configured successfully!


In [ ]:

class TicketState(BaseModel):
    ticket_text: str = ""
    category: str = ""
    priority: str = ""
    resolution: str = ""


In [ ]:

class CustomerSupportFlow(Flow[TicketState]):

    @start()
    def receive_ticket(self):
        ticket = input("Enter the customer's support issue: ").strip()

        if not ticket:
            raise ValueError("Support ticket cannot be empty.")

        self.state.ticket_text = ticket
        print("\nTicket received successfully!")
        print("Ticket:", self.state.ticket_text)

    @listen(receive_ticket)
    def categorize_ticket(self):
        prompt = f"""
        You are a customer support ticket classifier.

        Classify the following customer ticket into exactly ONE category:
        - billing: payments, refunds, invoices, charges
        - account_access: login problems, passwords, account lockouts
        - other: technical issues, feature requests, all other problems

        Return ONLY the category name.
        Do not explain your answer.

        Customer ticket:
        {self.state.ticket_text}
        """

        response = llm.call(prompt)
        category = str(response).strip().lower()

        if "billing" in category:
            category = "billing"
        elif "account_access" in category or "account access" in category:
            category = "account_access"
        else:
            category = "other"

        self.state.category = category

        print("\nTicket categorized!")
        print("Category:", self.state.category)

    @router(categorize_ticket)
    def route_by_category(self):
        category = self.state.category

        print(f"\nRouting ticket to: {category}")

        if category == "billing":
            return "billing"
        elif category == "account_access":
            return "account_access"
        else:
            return "other"

    @listen("billing")
    def handle_billing(self):
        self.state.priority = "High"
        self.state.resolution = (
            "Billing Support: Verify the transaction, review the invoice, "
            "and investigate any refund or incorrect charge."
        )
        print("Assigned to Billing Support.")

    @listen("account_access")
    def handle_access(self):
        self.state.priority = "High"
        self.state.resolution = (
            "Account Access Support: Help the customer reset their "
            "password, resolve login problems, or unlock their account."
        )
        print("Assigned to Account Access Support.")

    @listen("other")
    def handle_other(self):
        self.state.priority = "Medium"
        self.state.resolution = (
            "General Support: Investigate the technical issue or feature "
            "request and provide troubleshooting or escalation steps."
        )
        print("Assigned to General Support.")

    @listen(or_(handle_billing, handle_access, handle_other))
    def resolve_ticket(self):
        print("\n========== TICKET RESOLUTION ==========")
        print("Ticket:", self.state.ticket_text)
        print("Category:", self.state.category)
        print("Priority:", self.state.priority)
        print("Resolution:", self.state.resolution)
        print("=======================================")

        return {
            "ticket": self.state.ticket_text,
            "category": self.state.category,
            "priority": self.state.priority,
            "resolution": self.state.resolution
        }

support_flow = CustomerSupportFlow()
result = support_flow.kickoff()
print(result)






╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: CustomerSupportFlow                                                                                      │
│  ID: 5fa9686c-4376-46d0-9785-8f4a6259e675                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: CustomerSupportFlow                                                                                      │
│  ID: 5fa9686c-4376-46d0-9785-8f4a6259e675                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: receive_ticket                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Enter the customer's support issue: refund

Ticket received successfully!
Ticket: refund


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: receive_ticket                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: handle_billing                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: handle_billing                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: resolve_ticket                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: CustomerSupportFlow                                                                                      │
│  ID: 5fa9686c-4376-46d0-9785-8f4a6259e675                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'ticket': 'refund', 'category': 'billing', 'priority': 'High', 'resolution': 'Billing Support: Verify the transaction, review the invoice, and investigate any refund or incorrect charge.'}
